# Train QLoRA — Qwen2-VL-2B học từ 290 caption chưng cất

Cắt ra từ `scripts/kaggle_smoke.ipynb` (27 cell), chỉ giữ phần train.
18 cell đầu là benchmark của phiên trước — bỏ đi để tiết kiệm ~20 phút GPU mỗi lần chạy.


## 9. Huấn luyện QLoRA từ 500 caption chưng cất

Nạp `train.jsonl` (Phase 04, sinh bởi `scripts/dung_dataset_qlora.py`), gắn LoRA
vào Qwen2-VL-2B nén 4-bit, chỉ train tầng LLM (đóng băng vision encoder).

⚠️ **Kaggle mất sạch dữ liệu khi phiên tắt.** Sau khi train xong PHẢI bấm
**Save Version** và **tải file adapter zip về máy NGAY trong phiên** — đừng
hẹn "để lúc khác", phiên sau sẽ không còn gì.


In [ ]:
import os

# Chinh thong bao OOM goi y: gom cac khoi nho lai, do phan manh bo nho.
# Phai dat TRUOC khi torch cap phat lan dau moi an.
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

# Kaggle cap 2 card T4. Trainer thay 2 card la tu bat che do song song
# (DataParallel), nhan ban model ra ca hai -- nhung model da ghim o card 0 nen
# ban sao o card 1 doc vung nho khong phai cua no:
#   "AcceleratorError: Caught AcceleratorError in replica 0"
# Giau card 1 di ngay tu dau. Model 2B nen 4-bit chi ngon 5.2GB dinh, 1 card du.
os.environ['CUDA_VISIBLE_DEVICES'] = '0'  

import torch

print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHONG CO')

# Bay da gap: chon GPU trong Settings giua phien KHONG co tac dung len phien
# dang chay. Neu assert duoi day fail -> Stop session, chon lai Accelerator
# GPU T4, roi Start session lai tu dau (khong Restart, phai Stop han).
assert torch.cuda.is_available(), "Chua bat GPU: Settings > Accelerator > GPU T4, roi Stop session va chay lai"

# Trainer chi bat che do song song khi thay >1 card. Phai con dung 1.
print('So card torch nhin thay:', torch.cuda.device_count())
assert torch.cuda.device_count() == 1, (
    'Van thay nhieu card -> Trainer se nhan ban model va gay illegal memory access. '
    'CUDA_VISIBLE_DEVICES phai dat TRUOC khi import torch.'
)


In [ ]:
# Ghim transformers 4.x: ban 5.0.0 (pip -U keo ve) da doi ten lop model va
# rat co the doi ca cach giu bo nho -- notebook nay viet cho 4.x. Ghim de loai
# bien so nay ra khoi cuoc dieu tra OOM.
!pip install -q "transformers>=4.51,<5" peft bitsandbytes accelerate datasets
print('Cai xong')

import transformers
print('transformers:', transformers.__version__)


In [ ]:
from transformers import BitsAndBytesConfig, AutoProcessor

# Transformers ban moi doi ten AutoModelForVision2Seq -> AutoModelForImageTextToText.
# Cell tren chay 'pip install -U transformers' nen ban tren Kaggle khong co dinh;
# thu ten moi truoc, khong co thi lui ve ten cu.
try:
    from transformers import AutoModelForImageTextToText as AutoModelVLM
except ImportError:
    from transformers import AutoModelForVision2Seq as AutoModelVLM

QLORA_MODEL_HF_ID = 'Qwen/Qwen2-VL-2B-Instruct'  # khop vlm/model_registry.py key 'qwen2vl-2b'

# Cau hinh giong het vlm/model_loader.py:_tao_quant_config -- KHONG dung
# llm_int8_skip_modules: da do that tren T4, dung chung voi 4-bit gay loi
# "AssertionError: FP4 quantization state not initialized".
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float32,  # fp16 tran so tren QLoRA -> loss nan
    bnb_4bit_use_double_quant=True,
)

# T4 khong co bf16, va fp16 thi tran so tren QLoRA (loss thanh nan) -> chay fp32.
# Cham hon nhung dung. Model 2B nen 4-bit van chi ngon ~5GB dinh.
qlora_model = AutoModelVLM.from_pretrained(
    QLORA_MODEL_HF_ID,
    quantization_config=quant_config,
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True,
    # 'auto' chia model ra ca 2 card T4 -- nghe hay nhung khi train thi tensor
    # phai copy qua lai giua 2 card, moi lan chuyen tao ban sao tam, va gradient
    # checkpointing tinh lai activation nen chuyen RAT nhieu lan -> OOM o GPU 1.
    # Model 2B nen 4-bit chi ~1.8GB, nam gon 1 card 14.6GB thoai mai.
    device_map={'': 0},
    # Cach tinh chu y mac dinh + gradient checkpointing tren T4 (Turing) gay
    # "CUDA error: an illegal memory access". 'eager' cham hon chut nhung on dinh.
    attn_implementation='eager',
)
qlora_processor = AutoProcessor.from_pretrained(QLORA_MODEL_HF_ID)
print('Da nap model + processor:', QLORA_MODEL_HF_ID)

# Bang chung model nam gon 1 card: chi duoc phep co dung 1 thiet bi.
_thiet_bi = {str(ts.device) for ts in qlora_model.parameters()}
print('Model nam tren:', sorted(_thiet_bi))
assert len(_thiet_bi) == 1, f'Model bi chia ra nhieu card: {_thiet_bi}'
print(f'VRAM sau khi nap: {torch.cuda.memory_allocated(0)/1024**3:.2f} GB')


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Dong bang toan bo model truoc, gan LoRA se tu bat lai grad cho cac tang duoc nham.
for tham_so in qlora_model.parameters():
    tham_so.requires_grad = False

qlora_model = prepare_model_for_kbit_training(qlora_model, use_gradient_checkpointing=True)

# use_cache=True bao model GIU moi trang thai trung gian de sinh chu nhanh --
# nguoc han voi gradient checkpointing (VUT di roi tinh lai de tiet kiem). Bat ca
# hai thi phan giu thang: model 1.4GB ma 1 mau ngon 13GB. Phai tat tuong minh.
qlora_model.config.use_cache = False

# KHONG bat checkpointing thu cong cho vision: da thu, gay
# "CUDA error: an illegal memory access" khi Trainer bat them lan nua (chong nhau).
# Trainer tu lo qua gradient_checkpointing=True trong TrainingArguments.
if hasattr(qlora_model, 'generation_config'):
    qlora_model.generation_config.use_cache = False

# Chi nham tang chieu (projection) cua phan LLM -- KHONG dam vao vision tower.
# Non-goal cua ke hoach cam train vision: phan 'mat' dang doc dung vat the,
# cai sai la phan 'mieng' viet cau, dung dam vao mat.
LORA_TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=LORA_TARGET_MODULES,
)

qlora_model = get_peft_model(qlora_model, lora_config)

# Gradient checkpointing cat dut duong lan nguoc neu dau vao khong yeu cau grad
# -> train chay nhung model khong hoc gi. Bat lai tuong minh.
qlora_model.enable_input_require_grads()
qlora_model.print_trainable_parameters()

# Bang chung vision da dong bang: khong tang nao trong vision tower duoc train.
# Kiem thang bang ten tang -- chac hon dem tham so, vi p.numel() tren model nen
# 4-bit tra ve so O NHO (2 tham so gop 1 byte) chu khong phai so tham so that,
# lam mau so bot ~1.8 lan va ty le phong len gap doi.
_tang_train = [ten for ten, ts in qlora_model.named_parameters() if ts.requires_grad]
_vision_dinh = [ten for ten in _tang_train if 'visual' in ten or 'vision' in ten]
print(f'So tang duoc train: {len(_tang_train)}')
assert not _vision_dinh, f'LoRA nham vao vision tower: {_vision_dinh[:5]}'

# Ty le lay tu PEFT (get_nb_trainable_parameters da cong bu phan nen 4-bit).
_train_duoc, _tong = qlora_model.get_nb_trainable_parameters()
_ty_le = _train_duoc / _tong
print(f'Ty le tham so train duoc: {_ty_le:.4%}  ({_train_duoc:,} / {_tong:,})')
assert _ty_le < 0.01, f'Train qua nhieu tang: {_ty_le:.4%} >= 1%'


In [ ]:
import json
from pathlib import Path

import torch
from PIL import Image

# Kaggle Dataset gan qua Add Data -- duong dan thuc te tuy ten dataset da upload,
# sua DATASET_DIR neu khac. Dataset phai o che do Private (chua keyframe cuoc thi).
DATASET_DIR = Path('/kaggle/input/aic-vlm-distill-290')
TRAIN_JSONL = DATASET_DIR / 'train.jsonl'
IMAGES_DIR = DATASET_DIR / 'images'

def doc_jsonl(duong_dan):
    return [json.loads(dong) for dong in duong_dan.read_text(encoding='utf-8').splitlines()]

mau_train = doc_jsonl(TRAIN_JSONL)
mau_eval = doc_jsonl(DATASET_DIR / 'eval.jsonl')
print('So mau train:', len(mau_train), '| eval:', len(mau_eval))


def collate_qlora(batch):
    """Ghep anh + hoi thoai qua processor cua Qwen2-VL thanh 1 batch tensor."""
    anh_list = [Image.open(IMAGES_DIR / m['image']).convert('RGB') for m in batch]
    text_list = [
        qlora_processor.apply_chat_template(m['messages'], tokenize=False, add_generation_prompt=False)
        for m in batch
    ]
    enc = qlora_processor(text=text_list, images=anh_list, return_tensors='pt', padding=True)

    # Loss chi tinh tren phan model phai viet ra. Copy nguyen input_ids thi model
    # hoc thuoc ca cau hoi lan token anh -- loang tin hieu hoc.
    labels = enc['input_ids'].clone()
    labels[enc['attention_mask'] == 0] = -100
    for ten in ('image_token_id', 'video_token_id'):
        ma = getattr(qlora_processor, ten, None) or getattr(
            qlora_model.config, ten, None
        )
        if ma is not None:
            labels[labels == ma] = -100

    # Phan assistant bat dau sau moc nay; moi thu truoc do la de bai -> che di.
    moc = qlora_processor.tokenizer(
        '<|im_start|>assistant\n', add_special_tokens=False
    )['input_ids']
    moc_t = torch.tensor(moc, device=labels.device)
    for i in range(labels.size(0)):
        hang = enc['input_ids'][i]
        vi_tri = None
        for j in range(hang.size(0) - len(moc) + 1):
            if torch.equal(hang[j:j + len(moc)], moc_t):
                vi_tri = j + len(moc)
        if vi_tri is not None:
            labels[i, :vi_tri] = -100

    enc['labels'] = labels
    return enc


In [ ]:
# Bao cao bo nho chinh thuc cua PyTorch -- khong doan nua, doc so that.
import gc

torch.cuda.empty_cache(); gc.collect()
torch.cuda.reset_peak_memory_stats()

qlora_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
assert qlora_model.config.use_cache is False, 'use_cache van bat'

# Checkpointing CHI chay khi model o che do train. prepare_model_for_kbit_training
# co the da chuyen sang eval -> goi thang model (khong qua Trainer) thi checkpointing
# im lang khong hoat dong, va cell kiem ngon bo nho gap nhieu lan train that.
qlora_model.train()
_lop_co_cp = sum(1 for m in qlora_model.modules()
                 if getattr(m, 'gradient_checkpointing', False))
print(f'model.training = {qlora_model.training} | so module bat checkpointing = {_lop_co_cp}')
assert _lop_co_cp > 0, 'Khong module nao bat checkpointing -> se ngon bo nho gap nhieu lan'

_thu = {k: v.to(qlora_model.device) for k, v in collate_qlora([mau_train[0]]).items()}
print('input_ids:', tuple(_thu['input_ids'].shape),
      '| pixel_values:', tuple(_thu['pixel_values'].shape))
print(f'Truoc forward: {torch.cuda.memory_allocated()/1024**3:.2f} GB')

# Bat theo doi tung phep cap phat -- ghi lai ai xin bao nhieu, o dong nao.
torch.cuda.memory._record_memory_history(max_entries=100_000)

_loi = None
try:
    _ra = qlora_model(**_thu)
    print(f'Sau forward: {torch.cuda.memory_allocated()/1024**3:.2f} GB')
    print(f'  logits: {tuple(_ra.logits.shape)} -> '
          f'{_ra.logits.numel() * _ra.logits.element_size() / 1024**3:.2f} GB')
    _loss = float(_ra.loss)
    print(f'  loss = {_loss:.4f}')
    _ra.loss.backward()
    print(f'Sau backward: {torch.cuda.memory_allocated()/1024**3:.2f} GB')
except torch.cuda.OutOfMemoryError as e:
    _loi = e
    print(f'OOM -- dinh truoc khi vo: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB')

torch.cuda.memory._record_memory_history(enabled=None)

# Bang tong hop: khoi nao giu bao nhieu, phan manh ra sao.
print()
print(torch.cuda.memory_summary(abbreviated=True))

if _loi is not None:
    raise SystemExit('Da thu duoc bao cao bo nho o tren -- doc de biet cho ngon.')

_dinh = torch.cuda.max_memory_allocated() / 1024**3
print(f'DINH: {_dinh:.2f} GB')
qlora_model.zero_grad(set_to_none=True)
del _ra, _thu
torch.cuda.empty_cache(); gc.collect()
torch.cuda.reset_peak_memory_stats()
print('Duong ong OK -- bat dau train duoc.')


In [ ]:
from transformers import TrainingArguments, Trainer
from torch.utils.data import Dataset as TorchDataset


class QloraDataset(TorchDataset):
    def __init__(self, mau_list):
        self.mau_list = mau_list

    def __len__(self):
        return len(self.mau_list)

    def __getitem__(self, idx):
        return self.mau_list[idx]


training_args = TrainingArguments(
    output_dir='/kaggle/working/lora-qwen2vl-2b-haiku500',
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,   # bu batch nho -- 14.6GB khong cho batch lon voi anh
    num_train_epochs=2,              # >3 epoch tren 450 mau de hoc thuoc long thay vi hoc phong cach
    learning_rate=1e-4,
    # fp16 tren QLoRA hay tran so: loss thanh nan ngay buoc dau -> Trainer bao
    # train_loss=0.0 va eval_loss=nan (da gap that o v14, adapter vo dung).
    # T4 khong co bf16 nen chay fp32: cham hon nhung khong tran.
    fp16=False,
    max_grad_norm=0.3,               # cat grad lon, chan nan tu goc
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    optim='paged_adamw_8bit',
    save_strategy='epoch',
    eval_strategy='epoch',
    # Trainer mac dinh xoa cot khong khop chu ky forward cua model -- 'image' bi
    # xoa truoc khi toi collate_qlora -> KeyError. Collate tu lo viec dung tensor
    # tu mau tho nen phai giu nguyen cot.
    remove_unused_columns=False,
    per_device_eval_batch_size=1,
    logging_steps=1,                 # de chot chan nan kich hoat ngay buoc dau
    report_to='none',
)

trainer = Trainer(
    model=qlora_model,
    args=training_args,
    train_dataset=QloraDataset(mau_train),
    eval_dataset=QloraDataset(mau_eval),
    data_collator=collate_qlora,
)

from transformers import TrainerCallback


class ChanNan(TrainerCallback):
    """Dung ngay khi loss thanh nan.

    v14 chay het 35 phut moi lo ra loss nan -> adapter vo dung. Bat o buoc dau
    thi biet trong 1 phut."""

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs or 'loss' not in logs:
            return
        gia_tri = logs['loss']
        if gia_tri != gia_tri or gia_tri == 0.0:
            raise RuntimeError(
                f'Loss bat thuong o buoc {state.global_step}: {gia_tri}. '
                'Dung ngay -- train tiep chi phi GPU, adapter se vo dung.'
            )


trainer.add_callback(ChanNan())
ket_qua_train = trainer.train()
print(ket_qua_train)

ket_qua_eval = trainer.evaluate()
print('EVAL:', ket_qua_eval)
# Loss phai giam dan qua cac logging_steps o tren. Loss dung im hoac thanh nan
# -> dung ngay, kiem lai dtype (phai float16, khong bf16) va learning_rate.


In [ ]:
import shutil

ADAPTER_DIR = Path('/kaggle/working/lora-qwen2vl-2b-haiku500')
qlora_model.save_pretrained(ADAPTER_DIR)
qlora_processor.save_pretrained(ADAPTER_DIR)

ZIP_PATH = shutil.make_archive('/kaggle/working/lora-qwen2vl-2b-haiku500', 'zip', ADAPTER_DIR)
print('Da nen adapter:', ZIP_PATH)

# BAM 'Save Version' NGAY BAY GIO roi tai file zip o tab Output ve may.
# Kaggle mat sach /kaggle/working khi phien tat -- doi sang phien sau la mat trang.


## Checklist sau khi train

1. Tải file `lora-qwen2vl-2b-haiku500.zip` về máy (tab Output, sau khi Save Version).
2. Giải nén, đặt nội dung vào `results/lora-qwen2vl-2b-haiku500/` ở repo local.
3. Kiểm thư mục có đủ `adapter_config.json` + `adapter_model.safetensors`.
4. Chạy Phase 05 để đo trước/sau (nạp adapter vào `vlm/adapters.py`, so sánh với
   tập giữ lại `results/danh_sach_anh_holdout.txt`).
